# 03. Stage 2: Human-Supervised Sentiment Fine-Tuning
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)
### Academic Research Project: **GeoFinBERT**

---
### Research Methodology:
1. **Target**: Predict sentiment (`positive: 0`, `negative: 1`, `neutral: 2`) on real financial & commodity news.
2. **Ground Truth Integrity**: Exclusively independently human-labeled real articles. Zero synthetic labels.
3. **Partitioning**: Exact same chronological train, validation, and test splits for all models.
4. **Class Imbalance Mitigation**: Balanced inverse-frequency class-weighted cross-entropy loss:
   $$w_c = \frac{N}{K \cdot N_c}$$
5. **Head-to-Head Setup**:
   - **Model A (Baseline)**: Fine-tune original `ProsusAI/FinBERT`.
   - **Model B (Adapted)**: Fine-tune `GeoFinBERT-DAPT` (transferred encoder from Stage 1).
6. **Selection**: Champion classifier is saved as `GeoFinBERT`.


In [ ]:
# Setup & Environment Diagnostics
import os
import sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("[Colab Environment Detected] Installing dependencies...")
    !pip install -q -r requirements.txt
    !pip install -q -e .

import torch
import pandas as pd
from torch.utils.data import DataLoader

sys.path.append(os.path.abspath("src"))

from geofinbert.config import load_config
from geofinbert.utils import set_seed, get_device_info, count_parameters
from geofinbert.models.classifier import build_classifier_model
from geofinbert.data.dataset import SentimentClassificationDataset
from geofinbert.training.trainer_classifier import compute_class_weights, run_classifier_training

config = load_config("configs/config.yaml")
set_seed(config["project"]["seed"])
device_info = get_device_info()
print("Hardware Check:", device_info)
if device_info["cuda_available"]:
    print(f"Accelerated GPU Active: {device_info['gpu_name']} ({device_info['vram_gb']} GB VRAM)")


## Step 1: Load Chronological Human-Labeled Partitions


In [ ]:
splits_dir = config["data"]["splits_dir"]
train_df = pd.read_parquet(os.path.join(splits_dir, "sentiment_train.parquet"))
val_df = pd.read_parquet(os.path.join(splits_dir, "sentiment_val.parquet"))
test_df = pd.read_parquet(os.path.join(splits_dir, "sentiment_test.parquet"))

print(f"Labeled Splits: Train={len(train_df):,} | Val={len(val_df):,} | Test={len(test_df):,}")


## Step 2: Compute Balanced Class Weights
To prevent the model from biasing toward dominant classes, we calculate inverse-frequency class weights on the training partition.


In [ ]:
class_weights = compute_class_weights(train_df["label"].tolist(), num_classes=3)
print(f"Normalized Class Weights: {class_weights}")


## Step 3: Experiment A — Train Baseline FinBERT
We fine-tune the original `ProsusAI/FinBERT` on the chronological train partition.


In [ ]:
ft_cfg = config["stage2_finetuning"]
max_seq_len = config["model"]["max_seq_length"]
batch_size = ft_cfg["per_device_train_batch_size"]
eval_batch_size = ft_cfg["per_device_eval_batch_size"]

# Build Baseline Classifier
base_model, tokenizer = build_classifier_model(
    backbone_path_or_name=config["model"]["base_model_name"],
    num_labels=3,
    class_weights=class_weights,
    is_dapt_checkpoint=False,
    local_files_only=False
)

train_ds = SentimentClassificationDataset(train_df["text"].tolist(), train_df["label"].tolist(), tokenizer, max_length=max_seq_len)
val_ds = SentimentClassificationDataset(val_df["text"].tolist(), val_df["label"].tolist(), tokenizer, max_length=max_seq_len)
test_ds = SentimentClassificationDataset(test_df["text"].tolist(), test_df["label"].tolist(), tokenizer, max_length=max_seq_len)

num_workers = 2 if device_info["cuda_available"] else 0
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers)
val_loader = DataLoader(val_ds, batch_size=eval_batch_size, shuffle=False, num_workers=num_workers)
test_loader = DataLoader(test_ds, batch_size=eval_batch_size, shuffle=False, num_workers=num_workers)

baseline_log = run_classifier_training(
    model=base_model,
    tokenizer=tokenizer,
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    test_dataloader=test_loader,
    output_dir=ft_cfg["baseline_output_dir"],
    model_name="FinBERT-Baseline",
    num_epochs=ft_cfg["num_train_epochs"],
    learning_rate=float(ft_cfg["learning_rate"]),
    early_stopping_patience=ft_cfg["early_stopping_patience"],
    use_fp16=ft_cfg["fp16"] and device_info["cuda_available"]
)


## Step 4: Experiment B — Train Adapted GeoFinBERT
We fine-tune `GeoFinBERT-DAPT` (transferred domain-adapted backbone) using the exact same hyperparameters and chronological partitions.


In [ ]:
dapt_dir = config["stage1_dapt"]["output_dir"]

adapted_model, tokenizer = build_classifier_model(
    backbone_path_or_name=dapt_dir,
    num_labels=3,
    class_weights=class_weights,
    is_dapt_checkpoint=True,
    local_files_only=True
)

adapted_log = run_classifier_training(
    model=adapted_model,
    tokenizer=tokenizer,
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    test_dataloader=test_loader,
    output_dir=ft_cfg["adapted_output_dir"],
    model_name="GeoFinBERT",
    num_epochs=ft_cfg["num_train_epochs"],
    learning_rate=float(ft_cfg["learning_rate"]),
    early_stopping_patience=ft_cfg["early_stopping_patience"],
    use_fp16=ft_cfg["fp16"] and device_info["cuda_available"]
)


## Step 5: Side-by-Side Validation F1 Comparison


In [ ]:
import matplotlib.pyplot as plt

b_hist = baseline_log["history"]
a_hist = adapted_log["history"]

plt.figure(figsize=(8, 4))
plt.plot([h["epoch"] for h in b_hist], [h["macro_f1"] for h in b_hist], "r--s", label="Baseline FinBERT Val Macro F1")
plt.plot([h["epoch"] for h in a_hist], [h["macro_f1"] for h in a_hist], "b-o", label="GeoFinBERT Val Macro F1")
plt.xlabel("Epoch")
plt.ylabel("Validation Macro F1")
plt.title("Fine-Tuning Convergence: Baseline FinBERT vs GeoFinBERT")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## Step 6: Final Classifier Checkpoint Verification
The final classifier is verified and stored in `models/checkpoints/GeoFinBERT` ready for test set evaluation and inference.
